<a href="https://colab.research.google.com/github/IzadoraGalarza/TP1_G5_IA/blob/Develop/Trabalho_IA_Etapa_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Etapa 1 — Análise exploratória dos dados (NSL-KDD)

**Objetivo:** caracterizar a versão de trabalho da base, identificar o desbalanceamento, examinar atributos categóricos, inconsistências, redundâncias e possíveis fontes de vazamento.

> **Contexto de execução.** Os resultados consolidados abaixo foram registrados na análise realizada sobre a versão de trabalho com 69.833 linhas. O arquivo de dados original não está embutido neste notebook; para reexecutar as verificações, configure `DATA_PATH` na célula seguinte. O notebook detecta nomes de colunas comuns, mas pode exigir o ajuste de `TARGET_COL` conforme o nome real da variável-alvo.

A base usada no projeto contém campos derivados e operacionais (`src_bytes_kb`, `dst_bytes_kb`, `total_bytes`, `tipo_alerta_soc`, `score_ids_legado`, `tempo_resposta_analista_min`) que não pertencem ao esquema canônico tradicional de 41 atributos preditores do NSL-KDD. Portanto, a versão analisada deve ser tratada como uma base derivada/enriquecida, até que sua proveniência seja confirmada.

## Síntese dos resultados já obtidos

| Verificação | Resultado | Como considerar |
|---|---:|---|
| Registros / colunas | 69.833 / 50 | Confirmar as dimensões pelo arquivo carregado (`df.shape`). |
| Classe normal | 65.754 (94,16%) | Classe majoritária. |
| Classe anomalia | 4.079 (5,84%) | Razão aproximada de 16,12 normais por anomalia. Não avaliar só por acurácia. |
| Valores ausentes | 292.542 (8,38% das células) | Investigar mecanismo de ausência por coluna e por classe. |
| `tempo_resposta_analista_min` | 63.804 ausentes (91,37%) | Forte sinal de ausência estrutural/operacional e potencial vazamento. |
| `src_bytes` | 27.906 ausentes (39,96%) | Imputação pode afetar atributos derivados. |
| Atributo constante | `num_outbound_cmds` | Remover dos preditores após registrar a constância. |
| Duplicatas exatas | 11.355 ocorrências sinalizadas | Confirmar convenção de contagem e sobreposição. |
| Atributos iguais com rótulo divergente | 1.702 grupos / 3.730 linhas | Auditar origem e regra do rótulo antes de treinar. |
| Registros com ao menos um tipo de problema | 5.650 (8,09%) | Contagens por categoria podem se sobrepor. |

## Contexto e referência

O NSL-KDD foi proposto como uma revisão do KDD Cup 1999 para reduzir problemas associados à redundância de registros. O benchmark é histórico e não representa automaticamente o tráfego de redes contemporâneas. Referências: [página institucional da UNB/CIC](https://www.unb.ca/cic/datasets/nsl.html) e Tavallaee et al. (2009), [DOI: 10.1109/CISDA.2009.5356528](https://doi.org/10.1109/CISDA.2009.5356528).


## 1. Configuração e carregamento da base

Ajuste `DATA_PATH` para o arquivo utilizado no projeto. Se a classe tiver nome diferente dos nomes comuns, informe-o em `TARGET_COL`. A leitura tenta inferir o separador; se o arquivo for ARFF ou exigir parâmetros especiais, adapte a célula de leitura.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

DATA_PATH = Path("base_nsl_kdd.csv")  # <-- ajuste para o caminho do arquivo real
TARGET_COL = None                     # ex.: "classe", "class" ou "label"

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH, sep=None, engine="python")
    print(f"Arquivo: {DATA_PATH.resolve()}")
    print(f"Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas".replace(",", "."))
    display(df.head())
else:
    df = None
    print(f"Arquivo não encontrado: {DATA_PATH.resolve()}\nEdite DATA_PATH e execute novamente as células.")

Arquivo não encontrado: /content/base_nsl_kdd.csv
Edite DATA_PATH e execute novamente as células.


## 2. Inspeção estrutural e identificação da classe-alvo

In [ ]:
if df is not None:
    display(pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "não_nulos": df.notna().sum(),
        "ausentes": df.isna().sum(),
        "distintos": df.nunique(dropna=True)
    }).sort_values(["ausentes", "distintos"], ascending=[False, True]))

    candidatos = ["classe", "class", "label", "target", "classe_alvo", "target_class", "is_anomaly", "anomalia"]
    if TARGET_COL is None:
        encontradas = [c for c in candidatos if c in df.columns]
        TARGET_COL = encontradas[0] if encontradas else None
    if TARGET_COL is None or TARGET_COL not in df.columns:
        print("Coluna-alvo não detectada. Defina TARGET_COL manualmente e execute esta célula novamente.")
        classe_eda = None
    else:
        print(f"Coluna-alvo configurada: {TARGET_COL}")
        valores = df[TARGET_COL].dropna().astype(str).str.strip().str.lower().unique().tolist()
        print("Valores distintos da classe (amostra até 30):", valores[:30])
        if any(v in {"normal", "normal.", "normal traffic", "normal_traffic"} for v in valores):
            def binarizar_classe(valor):
                if pd.isna(valor):
                    return np.nan
                s = str(valor).strip().lower()
                if s in {"normal", "normal.", "normal traffic", "normal_traffic"}:
                    return "Normal"
                return "Anomalia"
            classe_eda = df[TARGET_COL].map(binarizar_classe)
            print("A classe foi agregada em Normal vs. Anomalia: Normal mantém os rótulos normais; outros rótulos não ausentes são agregados como Anomalia.")
        else:
            classe_eda = None
            print("Não foi possível inferir com segurança a codificação binária. Configure explicitamente o mapeamento antes de calcular métricas por classe.")
else:
    classe_eda = None

## 3. Classe-alvo e desbalanceamento

In [ ]:
if df is not None and classe_eda is not None:
    dist = classe_eda.value_counts(dropna=False).rename_axis("classe").to_frame("registros")
    dist["percentual"] = dist["registros"] / len(df) * 100
    display(dist)
    if {"Normal", "Anomalia"}.issubset(set(classe_eda.dropna().unique())):
        normais = int((classe_eda == "Normal").sum())
        anomalias = int((classe_eda == "Anomalia").sum())
        print(f"Razão normal/anomalia: {normais/anomalias:.2f}:1")
    dist_plot = classe_eda.value_counts()
    ax = dist_plot.plot(kind="bar", figsize=(7, 4), title="Distribuição da classe-alvo")
    ax.set_xlabel("Classe"); ax.set_ylabel("Registros"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
else:
    print("Etapa condicionada à configuração correta da coluna-alvo.")

Etapa condicionada à configuração correta da coluna-alvo.


## 4. Valores ausentes e atributos constantes

In [ ]:
if df is not None:
    aus = df.isna().sum().to_frame("ausentes")
    aus["percentual_linhas"] = aus["ausentes"] / len(df) * 100
    aus = aus[aus["ausentes"] > 0].sort_values("ausentes", ascending=False)
    display(aus)
    print("Total de células ausentes:", int(df.isna().sum().sum()))
    constantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print("Atributos constantes:", constantes)
    if classe_eda is not None and "tempo_resposta_analista_min" in df.columns:
        resumo = pd.DataFrame({"classe": classe_eda, "valor": df["tempo_resposta_analista_min"]})
        out = resumo.groupby("classe", dropna=False).agg(
            total=("valor", "size"), ausentes=("valor", lambda s: int(s.isna().sum())),
            observados=("valor", lambda s: int(s.notna().sum())), mediana=("valor", "median")
        )
        out["percentual_ausente"] = out["ausentes"] / out["total"] * 100
        display(out)
else:
    print("Carregue a base para calcular ausências e constantes.")

Carregue a base para calcular ausências e constantes.


## 5. Duplicatas exatas e grupos com rótulos conflitantes

In [ ]:
if df is not None:
    print("Linhas duplicadas além da primeira ocorrência:", int(df.duplicated().sum()))
    if TARGET_COL is not None and TARGET_COL in df.columns:
        preditores = [c for c in df.columns if c != TARGET_COL]
        grupos = df.groupby(preditores, dropna=False, sort=False)[TARGET_COL].nunique(dropna=False)
        grupos_conflitantes = grupos[grupos > 1]
        print("Grupos com atributos iguais e classe distinta:", int(len(grupos_conflitantes)))
        # Marca cada linha cuja assinatura de atributos aparece com mais de um rótulo.
        conflito_por_linha = df.groupby(preditores, dropna=False, sort=False)[TARGET_COL].transform(
            lambda s: s.nunique(dropna=False)
        ) > 1
        print("Linhas envolvidas nos grupos conflitantes:", int(conflito_por_linha.sum()))
        display(df.loc[conflito_por_linha].head(10))
else:
    print("Carregue a base para calcular duplicatas.")

Carregue a base para calcular duplicatas.


## 6. Integridade numérica: faixas, negativos e fatores de escala

In [ ]:
if df is not None:
    limites = {"dst_host_count": (0, 255), "count": (0, 511), "srv_count": (0, 511)}
    resultados_limites = []
    for col, (minimo, maximo) in limites.items():
        if col in df.columns:
            x = pd.to_numeric(df[col], errors="coerce")
            invalidos = x.notna() & ((x < minimo) | (x > maximo))
            resultados_limites.append({"atributo": col, "mínimo": minimo, "máximo": maximo,
                                      "fora_da_faixa": int(invalidos.sum()), "valor_máximo": x.max()})
    display(pd.DataFrame(resultados_limites))

    taxas = ["dst_host_srv_diff_host_rate", "dst_host_srv_rerror_rate", "rerror_rate", "dst_host_same_srv_rate", "dst_host_diff_srv_rate"]
    resultados_taxas = []
    for col in taxas:
        if col in df.columns:
            x = pd.to_numeric(df[col], errors="coerce")
            m = x.notna() & ((x < 0) | (x > 1))
            resultados_taxas.append({"atributo": col, "fora_[0,1]": int(m.sum()), "mínimo": x.min(), "máximo": x.max()})
    display(pd.DataFrame(resultados_taxas))

    candidatas_negativos = ["duration", "src_bytes", "dst_bytes", "total_bytes", "src_bytes_kb", "dst_bytes_kb"]
    neg = []
    for col in candidatas_negativos:
        if col in df.columns:
            x = pd.to_numeric(df[col], errors="coerce")
            neg.append({"atributo": col, "negativos": int((x < 0).sum()), "mínimo": x.min()})
    display(pd.DataFrame(neg))

    for col, maximo in [("count", 511), ("srv_count", 511), ("dst_host_count", 255)]:
        if col in df.columns:
            x = pd.to_numeric(df[col], errors="coerce")
            fora = x[(x < 0) | (x > maximo)]
            if len(fora):
                linha = {"atributo": col, "fora_da_faixa": len(fora)}
                for fator in [10, 100, 1000, 10000]:
                    ajustado = fora / fator
                    linha[f"caber_após_dividir_por_{fator}"] = int(((ajustado >= 0) & (ajustado <= maximo)).sum())
                display(pd.DataFrame([linha]))
else:
    print("Carregue a base para avaliar os limites numéricos.")

Carregue a base para avaliar os limites numéricos.


## 7. Consistência entre atributos derivados de bytes

In [ ]:
if df is not None:
    relacoes = [
        ("src_bytes", "src_bytes_kb", "src_bytes = src_bytes_kb × 1024"),
        ("dst_bytes", "dst_bytes_kb", "dst_bytes = dst_bytes_kb × 1024")
    ]
    for original, kb, descricao in relacoes:
        if original in df.columns and kb in df.columns:
            a = pd.to_numeric(df[original], errors="coerce")
            b = pd.to_numeric(df[kb], errors="coerce")
            ok = a.notna() & b.notna()
            inconsistentes = ok & ~np.isclose(a, b * 1024, rtol=0, atol=1e-9)
            print(f"{descricao}: {int(ok.sum())} pares comparáveis; {int(inconsistentes.sum())} inconsistências")
    if all(c in df.columns for c in ["total_bytes", "src_bytes", "dst_bytes"]):
        t = pd.to_numeric(df["total_bytes"], errors="coerce")
        s = pd.to_numeric(df["src_bytes"], errors="coerce")
        d = pd.to_numeric(df["dst_bytes"], errors="coerce")
        ok = t.notna() & s.notna() & d.notna()
        inconsistentes = ok & ~np.isclose(t, s + d, rtol=0, atol=1e-9)
        print(f"total_bytes = src_bytes + dst_bytes: {int(ok.sum())} pares comparáveis; {int(inconsistentes.sum())} inconsistências")
else:
    print("Carregue a base para testar as relações entre bytes.")

Carregue a base para testar as relações entre bytes.


## 8. Contagem por linha e comparação de qualidade por classe

In [ ]:
if df is not None:
    rate_cols = [c for c in df.columns if "rate" in c.lower()]
    counter_limits = {"dst_host_count": (0, 255), "count": (0, 511), "srv_count": (0, 511)}
    negative_cols = [c for c in ["duration", "src_bytes", "dst_bytes", "total_bytes", "src_bytes_kb", "dst_bytes_kb"] if c in df.columns]

    taxa_ruim = pd.Series(False, index=df.index)
    for col in rate_cols:
        x = pd.to_numeric(df[col], errors="coerce")
        taxa_ruim |= x.notna() & ((x < 0) | (x > 1))

    contador_ruim = pd.Series(False, index=df.index)
    for col, (lo, hi) in counter_limits.items():
        if col in df.columns:
            x = pd.to_numeric(df[col], errors="coerce")
            contador_ruim |= x.notna() & ((x < lo) | (x > hi))

    negativo = pd.Series(False, index=df.index)
    for col in negative_cols:
        x = pd.to_numeric(df[col], errors="coerce")
        negativo |= x.notna() & (x < 0)

    flags = pd.DataFrame({"taxas_fora_faixa": taxa_ruim, "contadores_fora_limite": contador_ruim, "valores_negativos": negativo})
    flags["numero_tipos"] = flags.sum(axis=1)
    display(flags["numero_tipos"].value_counts().sort_index().rename_axis("tipos_sinalizados").to_frame("registros"))

    if classe_eda is not None:
        resumo_classe = pd.DataFrame({"classe": classe_eda}).join(flags.drop(columns="numero_tipos"))
        linhas = []
        for classe, grupo in resumo_classe.groupby("classe", dropna=False):
            total = len(grupo)
            for nome in ["taxas_fora_faixa", "contadores_fora_limite", "valores_negativos"]:
                n = int(grupo[nome].sum())
                linhas.append({"classe": classe, "tipo": nome, "registros_sinalizados": n,
                               "total_classe": total, "percentual_na_classe": n / total * 100 if total else np.nan})
        display(pd.DataFrame(linhas))
else:
    print("Carregue a base para calcular a sobreposição entre tipos de problemas.")

Carregue a base para calcular a sobreposição entre tipos de problemas.


## 9. Categóricas e normalização de capitalização

In [ ]:
if df is not None:
    linhas = []
    for col in ["protocol_type", "service", "flag"]:
        if col in df.columns:
            original = df[col].dropna().astype(str)
            normalizado = original.str.strip().str.lower()
            linhas.append({"atributo": col, "níveis_originais": original.nunique(),
                           "níveis_após_strip_lower": normalizado.nunique(),
                           "exemplos_originais": ", ".join(original.drop_duplicates().head(12).tolist())})
    display(pd.DataFrame(linhas))
else:
    print("Carregue a base para inspecionar as categorias.")

Carregue a base para inspecionar as categorias.


## 10. Triagem de possíveis vazamentos

In [ ]:
if df is not None and classe_eda is not None:
    if "tipo_alerta_soc" in df.columns:
        ct = pd.crosstab(df["tipo_alerta_soc"], classe_eda, margins=True, dropna=False)
        display(ct)
        # Regra descritiva: ALERTA -> Anomalia; demais valores -> Normal, apenas para reproduzir o diagnóstico.
        pred_regra = np.where(df["tipo_alerta_soc"].astype(str).str.strip().str.upper().eq("ALERTA"), "Anomalia", "Normal")
        mask = classe_eda.notna()
        acc = (pred_regra[mask] == classe_eda[mask]).mean()
        print(f"Acurácia descritiva da regra alerta: {acc:.4%}. Não é avaliação de modelo nem deve ser interpretada como generalização.")
    if "score_ids_legado" in df.columns:
        score = pd.to_numeric(df["score_ids_legado"], errors="coerce")
        display(pd.DataFrame({"classe": classe_eda, "score": score}).groupby("classe")["score"].describe(percentiles=[.75,.95]))
    if "tempo_resposta_analista_min" in df.columns:
        tmp = pd.DataFrame({"classe": classe_eda, "tempo": pd.to_numeric(df["tempo_resposta_analista_min"], errors="coerce")})
        display(tmp.groupby("classe").agg(total=("tempo","size"), ausentes=("tempo",lambda s:int(s.isna().sum())),
                                           observados=("tempo",lambda s:int(s.notna().sum())), mediana_observada=("tempo","median")))
else:
    print("Carregue a base e configure a classe-alvo para executar a triagem de vazamento.")

Carregue a base e configure a classe-alvo para executar a triagem de vazamento.


## 11. Diagnóstico consolidado e decisões

Com base nos resultados observados, as recomendações são:

1. **Não treinar ainda com todos os campos.** Confirmar a proveniência de `tipo_alerta_soc`, `score_ids_legado` e `tempo_resposta_analista_min`; mantê-los fora do baseline até provar disponibilidade no instante de previsão.
2. **Auditar rótulos conflitantes.** Inspecionar os 1.702 grupos com atributos coincidentes e classes divergentes (3.730 linhas envolvidas). Não relabelar automaticamente.
3. **Não corrigir fatores de escala por palpite.** Validar os limites de `count`, `srv_count`, `dst_host_count` e das taxas com o arquivo de origem e o histórico do pipeline.
4. **Tratar ausências por mecanismo.** `src_bytes` tem 39,96% de ausência; `tempo_resposta_analista_min` tem 91,37%. Não preencher indiscriminadamente com zero ou mediana sem estudar o significado.
5. **Registrar duplicatas e evitar contaminação entre partições.** Entender o critério de duplicidade e manter grupos relacionados na mesma partição treino/validação/teste.
6. **Normalizar categorias com rastreabilidade.** Padronizar capitalização em `protocol_type` e `flag`, preservando as colunas originais; `service` não mostrou reduções por caixa.
7. **Considerar redundância funcional.** `src_bytes_kb`, `dst_bytes_kb` e `total_bytes` são matematicamente derivados dos valores em bytes e da soma; escolher conscientemente os atributos candidatos.
8. **Avaliar o desbalanceamento de maneira adequada.** Relatar matriz de confusão, precisão, recall e F1 para anomalia, F1 macro e PR-AUC; a acurácia trivial seria 94,16% sem detectar anomalias.

### Conclusão

A análise exploratória caracteriza a versão de trabalho e identifica os principais riscos para o pré-processamento e a modelagem. A conclusão é que a base pode seguir para uma preparação controlada, desde que as correções sejam documentadas e os pontos críticos — especialmente rótulos conflitantes, campos operacionais e valores fora de escala — sejam investigados antes da avaliação de modelos.

## 12. Apêndice — resultados numéricos consolidados

| Verificação | Resultado |
|---|---|
| `dst_host_count` fora de [0,255] | 912 registros; máximo 256.000 |
| `count` fora de [0,511] | 464 registros; máximo 512.000 |
| `srv_count` fora de [0,511] | 507 registros; máximo 512.000 |
| `dst_host_srv_diff_host_rate` fora de [0,1] | 739 |
| `dst_host_srv_rerror_rate` fora de [0,1] | 714 |
| `rerror_rate` fora de [0,1] | 683 |
| `dst_host_same_srv_rate` fora de [0,1] | 670 |
| `dst_host_diff_srv_rate` fora de [0,1] | 623 |
| `duration` negativos | 358, dos quais 323 são -1 |
| `src_bytes` negativos | 189 |
| `dst_bytes` negativos | 0 |
| `total_bytes` negativos | 73 |
| Registros sem tipos de problema sinalizados | 64.183 (91,91%) |
| Registros com um tipo de problema | 5.531 (7,92%) |
| Registros com dois tipos de problema | 119 (0,17%) |
| Registros com ao menos um tipo | 5.650 (8,09%) |

**Nota de interpretação:** contagens por coluna e por grupo de regra podem se sobrepor. O total de linhas únicas afetadas deve ser calculado por linha, sem somar ocorrências de diferentes colunas.
